# Llama 3.2 3B Instruct

## Model Setup

In [1]:
%pip install "transformers>=4.40" "datasets" "accelerate" "peft" "trl" "bitsandbytes" sentencepiece

import os
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Grab token
hf_token = os.getenv("HF_TOKEN")

if hf_token is None:
	raise ValueError("HF_TOKEN missing from .env file!")

# Export so Transformers + huggingface_hub sees it
os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token
os.environ["HF_HOME"] = "/root/.cache/huggingface"  # optional

print("HuggingFace token loaded from .env")

Note: you may need to restart the kernel to use updated packages.
HuggingFace token loaded from .env


In [2]:
import torch

def get_device():
	if torch.backends.mps.is_available():
		print("Using Apple Silicon GPU (MPS)")
		return "mps"
	elif torch.cuda.is_available():
		print("Using CUDA GPU")
		return "cuda"
	else:
		print("Using CPU")
		return "cpu"

DEVICE = get_device()

Using Apple Silicon GPU (MPS)


In [3]:
import os

HF_CACHE_DIR = os.path.join(os.getcwd(), ".hf_cache")
HF_DATASETS_CACHE = os.path.join(HF_CACHE_DIR, "datasets")
HF_MODELS_CACHE = os.path.join(HF_CACHE_DIR, "models")

os.makedirs(HF_DATASETS_CACHE, exist_ok=True)
os.makedirs(HF_MODELS_CACHE, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_DATASETS_CACHE"] = HF_DATASETS_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_MODELS_CACHE

In [4]:
import os
import json
from datasets import load_dataset
from transformers import (
	AutoTokenizer,
	AutoModelForCausalLM,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
import torch

# --- Paths ---
POI_CARDS_PATH = "../canonical/poi_cards.jsonl"
SFT_PATH       = "training/chat_sft_natural.jsonl"
OUTPUT_DIR     = "checkpoints/Llama-3.2-3B-Instruct-lora"

os.makedirs("training", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"

print("Config loaded")

/opt/anaconda3/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Config loaded


In [5]:
from datasets import load_dataset

SFT_PATH = "chat_sft_natural.jsonl"

ds = load_dataset(
	"json",
	data_files={"train": SFT_PATH},
	cache_dir=os.environ.get("HF_DATASETS_CACHE", "./.hf_cache/datasets"),
)
ds

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 51
    })
})

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os

CACHE_DIR = os.environ.get("TRANSFORMERS_CACHE", "./.hf_cache/models")

tokenizer = AutoTokenizer.from_pretrained(
	MODEL_NAME,
	cache_dir=CACHE_DIR
)

if tokenizer.pad_token is None:
	tokenizer.pad_token = tokenizer.eos_token

if DEVICE == "cuda":
	model = AutoModelForCausalLM.from_pretrained(
		MODEL_NAME,
		device_map="auto",
		torch_dtype=torch.bfloat16,
		cache_dir=CACHE_DIR,
	)
elif DEVICE == "mps":
	# MPS cannot use bfloat16; prefers float16
	model = AutoModelForCausalLM.from_pretrained(
		MODEL_NAME,
		torch_dtype=torch.float16,
		cache_dir=CACHE_DIR,
	)
	model.to("mps")
else:  # CPU
	model = AutoModelForCausalLM.from_pretrained(
		MODEL_NAME,
		torch_dtype=torch.float32,
		cache_dir=CACHE_DIR,
	)

print(f"Model loaded on {DEVICE}")

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded on mps


In [7]:
def formatting_func(example):
	"""
	TRL 0.25.1 passes ONE example at a time:
	  example["messages"] is a list of {role, content} dicts.
	We return a single string.
	"""
	system = ""
	user = ""
	assistant = ""

	for msg in example["messages"]:
		role = msg.get("role", "")
		content = msg.get("content", "")
		if role == "system":
			system = content
		elif role == "user":
			user = content
		elif role == "assistant":
			assistant = content

	full = (
		f"<s>[SYSTEM]\n{system}\n[/SYSTEM]\n"
		f"[USER]\n{user}\n[/USER]\n"
		f"[ASSISTANT]\n{assistant}\n</s>"
	)
	return full

In [8]:
peft_config = LoraConfig(
	r=16,
	lora_alpha=32,
	lora_dropout=0.05,
	bias="none",
	task_type="CAUSAL_LM",
	target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

In [ ]:
# %pip install trl
import torch
from trl import SFTTrainer, SFTConfig

# ---------- Device detection ----------
if torch.cuda.is_available():
	DEVICE = "cuda"
elif torch.backends.mps.is_available():
	DEVICE = "mps"
else:
	DEVICE = "cpu"

is_cuda = DEVICE == "cuda"
is_mps = DEVICE == "mps"

print("Using device:", DEVICE)

if tokenizer.pad_token is None:
	tokenizer.pad_token = tokenizer.eos_token

sft_config = SFTConfig(
	output_dir=OUTPUT_DIR,
	per_device_train_batch_size=1,
	gradient_accumulation_steps=4 if is_mps else 8,
	num_train_epochs=3,
	learning_rate=2e-4,
	logging_steps=5,
	save_steps=50,
	save_total_limit=2,
	bf16=is_cuda,
	fp16=is_cuda, # don't use on MPS
	packing=True,
	max_length=1024 if is_mps else 2048,  # shorter on MPS to save memory
)

# helps with memory on GPU/MPS
if is_cuda or is_mps:
	model.gradient_checkpointing_enable()


trainer = SFTTrainer(
	model=model,
	args=sft_config,
	train_dataset=ds["train"],
	peft_config=peft_config, # LoRA config
	processing_class=tokenizer,
	formatting_func=formatting_func,  # must return str or list[str] per example
)

print("SFTTrainer is ready")


Using device: mps
'NoneType' object has no attribute 'cadam32bit_grad_fp32'


/opt/anaconda3/lib/python3.12/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "
Padding-free training is enabled, but the attention implementation is not set to a supported flash attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
You are using packing, but the attention imp

Tokenizing train dataset:   0%|          | 0/51 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/51 [00:00<?, ? examples/s]

SFTTrainer is ready


In [10]:
trainer.train()
trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)

print("Training complete; model saved to", OUTPUT_DIR)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.
/opt/anaconda3/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
5,1.399200
10,1.056400
15,0.806300
20,0.575700
25,0.396900
30,0.328500
35,0.287600


'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: b7cf66ff-4bd9-479a-83f8-a03eedbaa45c)')' thrown while requesting HEAD https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json
Retrying in 1s [Retry 1/5].


Training complete; model saved to checkpoints/Llama-3.2-3B-Instruct-lora


## Metrics

In [ ]:
import json
import math
from collections import defaultdict

# Path to your canonical POI data
POI_PATH = "../canonical/poi_cards.jsonl"

def norm_name(name: str) -> str:
    return (name or "").strip().lower()

NAME_INDEX = defaultdict(list)
with open(POI_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        poi = json.loads(line)
        disp = poi.get("name") or poi.get("_display_name")
        if not disp:
            continue
        NAME_INDEX[norm_name(disp)].append(poi)
    
def get_card_coords(card):
    coords = card.get("coordinates") or card.get("coords") or card.get("center")
    if coords:
        return coords["lat"], coords["lon"]
    return None, None

def load_poi_index(path=POI_PATH):
	index = {}
	with open(path, "r", encoding="utf-8") as f:
		for line in f:
			if not line.strip():
				continue
			poi = json.loads(line)
			name = poi.get("name") or poi.get("_display_name")
			if not name:
				continue
			key = name.strip().lower()
			index[key] = poi
	return index

poi_index = load_poi_index()
print(f"Loaded {len(poi_index)} POIs into index")

def get_card_category(card: dict):
    if "_category" in card:
        return card["_category"]
    tags = card.get("tags") or {}
    amenity = (tags.get("amenity") or "").lower()
    tourism = (tags.get("tourism") or "").lower()
    leisure = (tags.get("leisure") or "").lower()
    natural = (tags.get("natural") or "").lower()

    if tourism in ("museum", "gallery", "attraction", "theme_park", "zoo", "aquarium", "viewpoint"):
        return "attraction"
    if leisure in ("park", "garden"):
        return "park"
    if natural == "beach":
        return "beach"
    if amenity in ("restaurant", "cafe", "fast_food"):
        return "food"
    return "other"

def match_cards_for_stop(stop):
    return NAME_INDEX.get(norm_name(stop.get("name")), [])


# Helpers for metrics

def haversine_m(lat1, lon1, lat2, lon2):
    R = 6371000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    h = math.sin(dphi/2)**2 + math.cos(p1) * math.cos(p2) * math.sin(dlmb/2)**2
    return 2 * R * math.asin(math.sqrt(h))

## TDTS — TRAVEL DISTANCE / TIME SCORE

def compute_tdts(itinerary, max_travel_minutes: float, default_speed_kmh=20.0):
    if not itinerary or len(itinerary) < 2:
        return None

    total_dist_m = 0
    legs = 0

    for i in range(1, len(itinerary)):
        prev_cards = match_cards_for_stop(itinerary[i-1])
        curr_cards = match_cards_for_stop(itinerary[i])
        if not prev_cards or not curr_cards:
            continue

        plat, plon = get_card_coords(prev_cards[0])
        clat, clon = get_card_coords(curr_cards[0])
        if None in (plat, plon, clat, clon):
            continue

        d = haversine_m(plat, plon, clat, clon)
        total_dist_m += d
        legs += 1

    if legs == 0:
        return None

    dist_km = total_dist_m / 1000
    minutes = (dist_km / default_speed_kmh) * 60

    score = 1 - (minutes / max_travel_minutes)
    return max(0, min(1, score))


# Diversity Index

def compute_diversity_index(itinerary):
    if not itinerary:
        return None

    categories = []
    for stop in itinerary:
        cards = match_cards_for_stop(stop)
        cat = get_card_category(cards[0]) if cards else "unknown"
        categories.append(cat)

    return len(set(categories)) / len(itinerary)


# Preference Coverage

def compute_preference_coverage(itinerary, prefs):
    if not itinerary:
        return None
    if not prefs:
        return 1.0

    prefs_l = [p.lower() for p in prefs]

    def matches(stop):
        text = (stop.get("name","") + " " + stop.get("explanation","")).lower()
        return any(p in text for p in prefs_l)

    return sum(matches(s) for s in itinerary) / len(itinerary)


# Accessibility Coverage

def card_satisfies_constraint(card, constraint):
    tags = card.get("tags") or {}
    a11y = (card.get("a11y") or {}).get("combined", {})

    if constraint == "wheelchair":
        w = (tags.get("wheelchair") or "").lower()
        if w in ("yes", "designated"):
            return True
        if a11y.get("score", 0) >= 0.5:
            return True
        return False

    return False

def stop_satisfies_all_constraints(stop, constraints):
    cards = match_cards_for_stop(stop)
    if not constraints:
        return True
    if not cards:
        return False
    return any(all(card_satisfies_constraint(c, con) for con in constraints) for c in cards)

def stop_satisfies_constraint_count(stop, constraints):
    cards = match_cards_for_stop(stop)
    if not constraints or not cards:
        return 0
    best = 0
    for c in cards:
        cnt = sum(card_satisfies_constraint(c, con) for con in constraints)
        best = max(best, cnt)
    return best

def compute_access_accuracy(itinerary, constraints):
    if not itinerary:
        return None
    if not constraints:
        return 1.0
    ok = sum(stop_satisfies_all_constraints(s, constraints) for s in itinerary)
    return ok / len(itinerary)

def compute_compliance_rate(itinerary, constraints):
    if not itinerary:
        return None
    if not constraints:
        return 1.0
    N, C = len(itinerary), len(constraints)
    satisfied = sum(stop_satisfies_constraint_count(s, constraints) for s in itinerary)
    return satisfied / (N * C)

def compute_grounding_rate(itinerary):
    if not itinerary:
        return None
    total = len(itinerary)
    grounded = sum(1 for s in itinerary if match_cards_for_stop(s))
    return grounded / total



Loaded 2735 POIs into index


## Model + SFT (LoRA) + RAG

In [13]:
%pip install sentence-transformers

import json
import math
import os
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
from sentence_transformers import SentenceTransformer

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [ ]:
POI_PATH = "../canonical/poi_cards.jsonl"

def load_cards(path: str) -> List[Dict[str, Any]]:
    cards = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            cards.append(json.loads(line))
    print(f"[load_cards] loaded {len(cards)} POIs")
    return cards

cards = load_cards(POI_PATH)
len(cards)

[load_cards] loaded 4018 POIs


4018

In [19]:
# Helpers

def get_category(tags: dict) -> str:
    amenity = (tags.get("amenity") or "").lower()
    tourism = (tags.get("tourism") or "").lower()
    leisure = (tags.get("leisure") or "").lower()
    natural = (tags.get("natural") or "").lower()

    if tourism in ("museum", "gallery", "attraction", "theme_park", "zoo", "aquarium", "viewpoint"):
        return "attraction"
    if leisure in ("park", "garden"):
        return "park"
    if natural in ("beach",):
        return "beach"
    if amenity in ("restaurant", "cafe", "fast_food"):
        return "food"
    if tourism in ("hotel", "hostel"):
        return "lodging"
    return "other"


def get_display_name(card: dict) -> str:
    if card.get("name"):
        return card["name"]
    tags = card.get("tags") or {}
    return tags.get("name") or "(unnamed place)"


def annotate_cards(cards: List[Dict[str, Any]]) -> None:
    for c in cards:
        tags = c.get("tags") or {}
        c["_category"] = get_category(tags)

        a11y = c.get("a11y") or {}
        combined = a11y.get("combined") or {}
        c["_score"] = combined.get("score", 0.0)

        c["_display_name"] = get_display_name(c)

def get_coords(poi: dict) -> Tuple[Optional[float], Optional[float]]:

    coords = poi.get("coords") or poi.get("coordinates") or poi.get("center")
    if coords:
        return coords["lat"], coords["lon"]
    return None, None


def haversine_m(lat1, lon1, lat2, lon2) -> float:
    R = 6371000.0  # meters
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlmb = math.radians(lon2 - lon1)
    h = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlmb / 2) ** 2
    return 2 * R * math.asin(math.sqrt(h))

def describe_accessibility(card: dict, constraints: List[str]) -> str:
    tags = card.get("tags") or {}
    a11y = card.get("a11y") or {}
    combined = a11y.get("combined") or {}

    parts = []

    w = (tags.get("wheelchair") or "").lower()
    if w == "yes":
        parts.append("wheelchair accessible")
    elif w == "limited":
        parts.append("partially wheelchair accessible")
    elif w == "no":
        parts.append("not wheelchair accessible")

    if (tags.get("tactile_paving") or "").lower() == "yes":
        parts.append("tactile paving available")
    if (tags.get("hearing_loop") or "").lower() == "yes":
        parts.append("hearing loop available")
    if (tags.get("toilets:wheelchair") or "").lower() == "yes":
        parts.append("accessible restroom")

    grade = combined.get("grade")
    if grade is not None:
        parts.append(f"overall accessibility grade {grade}")

    best_mode = combined.get("best_mode")
    if best_mode:
        parts.append(f"best reached by {best_mode}")

    if not parts:
        parts.append("basic accessibility (few explicit tags available)")

    if constraints:
        parts.append("relevant for constraints: " + ", ".join(constraints))

    return ", ".join(parts)


annotate_cards(cards)
cards[0]

{'id': 'osm:node:150964201',
 'name': 'Easter Cross',
 'category': 'attraction',
 'coords': {'lat': 32.8398116, 'lon': -117.2446749},
 'source': 'OSM',
 'wheelchair': None,
 'entrance_wheelchair': None,
 'toilets_wheelchair': None,
 'surface': None,
 'tactile_paving': None,
 'ramp': None,
 'smoothness': None,
 'incline': None,
 'steps': None,
 'elevator': None,
 'nearest_stop_id': None,
 'nearest_stop_name': None,
 'distance_to_nearest_stop_m': None,
 'routes': [],
 'tags': {'alt_name': 'Mount Soledad Cross',
  'ele': '244',
  'gnis:feature_id': '1660587',
  'internet_access': 'wlan',
  'man_made': 'cross',
  'name': 'Easter Cross',
  'tourism': 'attraction',
  'wikidata': 'Q337916'},
 'nearest_stop': None,
 'nearest_parking': {'id': 'osm:way:100403424',
  'kind': 'parking',
  'distance_m': 26.9,
  'access': None,
  'fee': None,
  'capacity_disabled': None,
  'maxheight': None,
  'parking': 'street_side'},
 'a11y': {'onsite_score': 0.0,
  'reach': {'transit': {'score': 0.0, 'distance_m

### RAG Setup

In [20]:
EMB_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMB_CACHE_DIR = "./.hf_cache/embeddings"

embedder = SentenceTransformer(EMB_MODEL_NAME, cache_folder=EMB_CACHE_DIR)
print("Embedding model loaded.")

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 6567f750-e9f2-470b-b3f9-f3a50d8822ab)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


In [27]:
def poi_to_text(card: dict) -> str:
    name = card["_display_name"]
    cat = card["_category"]
    tags = card.get("tags") or {}
    a11y_desc = describe_accessibility(card, constraints=["wheelchair"])  # generic; fine for retrieval

    tag_bits = []
    for k in ("amenity", "tourism", "cuisine"):
        if tags.get(k):
            tag_bits.append(f"{k}={tags[k]}")
    tags_str = ", ".join(tag_bits) if tag_bits else "few tags"

    return f"{name}. Category: {cat}. Tags: {tags_str}. Accessibility: {a11y_desc}."


poi_texts = [poi_to_text(c) for c in cards]
len(poi_texts), poi_texts[0][:200]

(4018,
 'Easter Cross. Category: attraction. Tags: tourism=attraction. Accessibility: overall accessibility grade D, best reached by driving, relevant for constraints: wheelchair.')

In [28]:
# Compute and store embeddings
poi_embs = embedder.encode(poi_texts, batch_size=64, convert_to_numpy=True, normalize_embeddings=True)
poi_embs.shape

(4018, 384)

In [29]:
# index mapping
idx_to_card = {i: cards[i] for i in range(len(cards))}

In [ ]:
# Anchor Selector --> We will choose one anchor and compute distances to all candidate POIs

def find_poi_by_name(name: str) -> Optional[dict]:
    name_low = name.lower()
    for c in cards:
        if get_display_name(c).lower() == name_low:
            return c
    return None

def semantic_topk_poi_indices(query: str, top_k: int = 50) -> List[int]:
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    sims = poi_embs @ q_emb  # cosine similarity since both normalized
    top_idx = np.argsort(-sims)[:top_k]
    return top_idx.tolist()

import random
from typing import Optional, List, Dict, Any

def pick_anchor_poi(
    user_query: str,
    anchor_name: Optional[str] = None,
    wheelchair_only: bool = True,
    categories: Optional[List[str]] = None,
    top_k_anchor: int = 5,
    rng: Optional[random.Random] = None,
) -> Optional[Dict[str, Any]]:
    """
    Pick an anchor POI:
      - if anchor_name is provided, try exact (or fuzzy) match
      - otherwise: semantic search -> filter by wheelchair/category -> sample from top_k_anchor
    """
    if rng is None:
        rng = random

    # 1) explicit anchor name override
    if anchor_name is not None:
        name_norm = anchor_name.strip().lower()
        best = None
        best_score = -1.0
        for card in all_poi_cards:
            disp = (card.get("name") or card.get("_display_name") or "").strip().lower()
            if disp == name_norm:
                # optional: require wheelchair / category here too
                if wheelchair_only and not passes_constraints(card, wheelchair_only=True):
                    continue
                best = card
                best_score = 1.0
                break
        if best is not None:
            print(f"[anchor] using explicit anchor: {best['_display_name']} (category={best['_category']})")
            return best
        # if explicit name not found, fall through to semantic mode

    # semantic top-K candidates
    cats_filter = {c.lower() for c in categories} if categories else None
    top_idx = semantic_topk_poi_indices(user_query, top_k=top_k_anchor * 3)  # oversample then filter

    anchor_candidates: List[Dict[str, Any]] = []
    for idx in top_idx:
        card = idx_to_card[idx]

        if wheelchair_only and not passes_constraints(card, wheelchair_only=True):
            continue
        if cats_filter and card.get("_category") not in cats_filter:
            continue

        anchor_candidates.append(card)
        if len(anchor_candidates) >= top_k_anchor:
            break

    if not anchor_candidates:
        print("[anchor] no valid anchor candidates found")
        return None

    # randomize among top-K to get diversity
    anchor = rng.choice(anchor_candidates)
    print(f"[anchor] auto-picked anchor: {anchor['_display_name']} (category={anchor['_category']}) "
          f"from {len(anchor_candidates)} candidates")
    return anchor

In [ ]:
def passes_constraints(card: dict, wheelchair_only: bool) -> bool:
    tags = card.get("tags") or {}
    if wheelchair_only:
        w = (tags.get("wheelchair") or "").lower()
        if w not in ("yes", "designated"):
            return False
    return True

def retrieve_pois_rag(
    user_query: str,
    anchor_poi: dict,
    top_k_semantic: int = 80,
    top_k_final: int = 30,
    wheelchair_only: bool = True,
    categories: Optional[List[str]] = None,
    min_score: float = 0.0,
    max_dist_from_anchor_m: Optional[float] = None,
) -> List[dict]:
    cats_filter = {c.lower() for c in categories} if categories else None

    # semantic retrieval
    top_idx = semantic_topk_poi_indices(user_query, top_k=top_k_semantic)

    # anchor coords
    anchor_lat, anchor_lon = get_coords(anchor_poi)

    candidates = []
    for idx in top_idx:
        c = idx_to_card[idx]

        # accessibility + category filters
        if c["_score"] < min_score:
            continue
        if cats_filter and c["_category"] not in cats_filter:
            continue
        if not passes_constraints(c, wheelchair_only):
            continue

        # distance from anchor
        lat, lon = get_coords(c)
        if None not in (anchor_lat, anchor_lon, lat, lon):
            d_m = haversine_m(anchor_lat, anchor_lon, lat, lon)
        else:
            d_m = None

        # distance cutoff
        if max_dist_from_anchor_m is not None and d_m is not None:
            if d_m > max_dist_from_anchor_m:
                continue

        c = dict(c)
        c["_anchor_dist_m"] = d_m
        candidates.append(c)

    # sort: primarily by distance (if known), secondarily by a11y score
    def sort_key(c):
        d = c.get("_anchor_dist_m")
        d = d if d is not None else 1e9
        return (d, -c["_score"])

    candidates.sort(key=sort_key)
    if len(candidates) > top_k_final:
        candidates = candidates[:top_k_final]

    print(f"[rag] retrieved {len(candidates)} candidates around anchor {anchor_poi['_display_name']}")
    return candidates

In [ ]:
from typing import List, Optional, Dict, Any

def format_candidates_for_prompt(
    cands: List[Dict[str, Any]],
    constraints: List[str],
    anchor_poi: Dict[str, Any],
) -> str:
    """
    Format POIs for the LLM prompt, including:
      - category
      - coarse distance from anchor (using _anchor_dist_m)
      - natural-language accessibility description
    """
    lines = []
    anchor_name = anchor_poi["_display_name"]

    for c in cands:
        name = c["_display_name"]
        cat = c["_category"]
        a11y_desc = describe_accessibility(c, constraints)

        d_m = c.get("_anchor_dist_m")
        if d_m is not None:
            if d_m < 300:
                dist_str = "very close to the anchor"
            elif d_m < 1000:
                dist_str = f"about {int(d_m)} meters from the anchor"
            else:
                dist_str = f"about {int(d_m / 1000)} km from the anchor"
        else:
            dist_str = "distance from the anchor is unknown"

        lines.append(
            f"- {name}: category={cat}, {dist_str}, accessibility={a11y_desc}"
        )

    header = f"Anchor location: {anchor_name}\n\nCandidate places:\n"
    return header + "\n".join(lines)

def rag_pipeline_for_user(
    user_text: str,
    anchor_name: Optional[str] = None,
    wheelchair_only: bool = True,
    categories: Optional[List[str]] = None,
    min_score: float = 0.0,
    max_dist_from_anchor_m: Optional[float] = 6000,
    top_k_semantic: int = 80,
    top_k_final: int = 30,
):
    """
    High-level RAG orchestration:
      - pick anchor
      - retrieve RAG candidates
      - format them for the prompt
    """
    # pick anchor
    anchor = pick_anchor_poi(
        user_query=user_text,
        anchor_name=anchor_name,
        wheelchair_only=wheelchair_only,
    )
    if anchor is None:
        raise ValueError("Could not find anchor POI for this query.")

    # retrieve candidates around anchor
    cands = retrieve_pois_rag(
        user_query=user_text,
        anchor_poi=anchor,
        top_k_semantic=top_k_semantic,
        top_k_final=top_k_final,
        wheelchair_only=wheelchair_only,
        categories=categories,
        min_score=min_score,
        max_dist_from_anchor_m=max_dist_from_anchor_m,
    )

    # format candidate text for the LLM
    cand_text = format_candidates_for_prompt(
        cands,
        constraints=["wheelchair"] if wheelchair_only else [],
        anchor_poi=anchor,
    )
    return anchor, cands, cand_text

In [75]:
user_query = (
    "I use a wheelchair and I love art museums and coffee. "
    "I want a relaxed day in San Diego with short travel distances."
)

anchor, cands, cand_text = rag_pipeline_for_user(
    user_query,
    anchor_name=None,
    wheelchair_only=True,
    categories=["attraction", "food", "park"],
    min_score=0.2,
    max_dist_from_anchor_m=9000,
)

print("=== Anchor ===")
print(anchor["_display_name"], anchor["_category"])
print()
print("=== Candidate snippet for prompt ===")
print(cand_text[:1000])

[anchor] auto-picked anchor: San Diego Natural History Museum (category=attraction)
[rag] retrieved 4 candidates around anchor San Diego Natural History Museum
=== Anchor ===
San Diego Natural History Museum attraction

=== Candidate snippet for prompt ===
Anchor location: San Diego Natural History Museum

Candidate places:
- San Diego Natural History Museum: category=attraction, very close to the anchor, accessibility=wheelchair accessible, overall accessibility grade C, best reached by transit, relevant for constraints: wheelchair
- San Diego History Center: category=attraction, very close to the anchor, accessibility=wheelchair accessible, overall accessibility grade D, best reached by transit, relevant for constraints: wheelchair
- San Diego Zoo: category=attraction, about 536 meters from the anchor, accessibility=wheelchair accessible, overall accessibility grade D, best reached by transit, relevant for constraints: wheelchair
- SeaWorld San Diego: category=attraction, about 8 km 

## Inference + Evaluation

In [60]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE != "cpu" else torch.float32,
    cache_dir=CACHE_DIR,
)
if DEVICE != "cpu":
    base_model.to(DEVICE)

tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ft_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
ft_model.eval()

print("Loaded base + LoRA fine-tuned model.")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded base + LoRA fine-tuned model.


In [61]:
SYSTEM_PROMPT = (
    "You are an accessibility-aware itinerary planner for San Diego. "
    "You must obey all accessibility constraints, be honest about the data, "
    "and prefer itineraries with short travel distances between stops."
)

In [64]:
import json
import torch

def generate_itinerary_with_rag(
    user_query: str,
    anchor_name: Optional[str] = None,
    max_new_tokens: int = 512,
    temperature: float = 0.7,
    top_p: float = 0.9,
):
    # RAG: anchor + candidates
    anchor, cands, cand_text = rag_pipeline_for_user(
        user_text=user_query,
        anchor_name=anchor_name,
        wheelchair_only=False,
        categories=["attraction", "park", "food"],
        min_score=0.2,
        max_dist_from_anchor_m=9000,
        top_k_semantic=80,
        top_k_final=25,
    )

    user_msg = (
        "You are a helpful assistant that plans one-day itineraries in San Diego.\n\n"
        "You MUST:\n"
        "- Use ONLY the places listed below (do NOT invent new locations).\n"
        "- Respect all accessibility needs.\n"
        "- Prefer short travel distances between consecutive stops.\n"
        "- Produce a realistic schedule with non-overlapping times.\n\n"
        "Here is the traveler description:\n"
        f"{user_query}\n\n"
        "Accessibility needs:\n"
        "- wheelchair\n\n"
        f"{cand_text}\n\n"
        "Using only these places, create a coherent one-day itinerary. "
        "Make sure the itinerary is realistic in terms of timing and explicitly respects the accessibility needs.\n\n"
        "Return your answer as a JSON object with a single field \"itinerary\", where the value is a list of stops. "
        "Each stop must have:\n"
        "- \"name\": the place name\n"
        "- \"start_time\": e.g., \"10:00\"\n"
        "- \"end_time\": e.g., \"11:30\"\n"
        "- \"explanation\": a short explanation of why this stop is a good fit, "
        "including accessibility reasoning and, when relevant, short travel distance from the previous stop.\n"
    )

    # Match SFT formatting
    prompt = (
        f"<s>[SYSTEM]\n{SYSTEM_PROMPT}\n[/SYSTEM]\n"
        f"[USER]\n{user_msg}\n[/USER]\n"
        f"[ASSISTANT]\n"
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    )

    if DEVICE != "cpu":
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            eos_token_id=tokenizer.eos_token_id,
        )

    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=False)

    assistant_part = full_text.split("[ASSISTANT]")[-1]
    
    start_idx = assistant_part.find("{")
    end_idx = assistant_part.rfind("}")
    json_str = None
    parsed = None
    if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
        json_str = assistant_part[start_idx : end_idx + 1]
        try:
            parsed = json.loads(json_str)
        except Exception:
            parsed = None

    return {
        "anchor": anchor,
        "candidates": cands,
        "prompt": prompt,
        "raw_output": full_text,
        "json_str": json_str,
        "parsed": parsed,
    }

In [65]:
user_query = (
    "I use a wheelchair and I love art museums and coffee. "
    "I want a relaxed, accessible one-day itinerary in San Diego from 10:00 to 18:00 "
)

result = generate_itinerary_with_rag(user_query)

print("=== Anchor ===")
print(result["anchor"]["_display_name"], "-", result["anchor"]["_category"])
print()

print("=== Parsed itinerary ===")
print(result["parsed"])

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


[anchor] auto-picked anchor: San Diego Museum of Art (category=attraction)
[rag] retrieved 25 candidates around anchor San Diego Museum of Art
=== Anchor ===
San Diego Museum of Art - attraction

=== Parsed itinerary ===
{'itinerary': [{'name': 'San Diego Museum of Art', 'start_time': '10:00', 'end_time': '11:42', 'explanation': "This stop is the anchor and a good starting point for the day. It matches the user's accessibility needs because it is accessible overall accessibility grade D, best reached by driving, relevant for constraints: wheelchair."}, {'name': 'San Diego History Center', 'start_time': '11:42', 'end_time': '13:24', 'explanation': "This stop is approximately 27 meters from the previous stop. It matches the user's accessibility needs because it is accessible overall accessibility grade D, best reached by transit, relevant for constraints: wheelchair."}, {'name': 'San Diego Natural History Museum', 'start_time': '13:24', 'end_time': '15:06', 'explanation': "This stop is a

In [95]:
import json
import random
from typing import List, Dict, Any
import torch
import pandas as pd

# --------------------------------------------------
# 1) Evaluation scenarios
# --------------------------------------------------
EVAL_SCENARIOS = [
    {
        "name": "wheelchair_museums_coffee",
        "user_text": (
            "I use a wheelchair and I love art museums and coffee. "
            "I want a relaxed, accessible one-day itinerary in San Diego from 10:00 to 18:00 "
            "with a mix of culture and food."
        ),
        "constraints": [],
        "prefs": ["museum", "art", "coffee"],
        "categories": ["attraction", "food", "park"],
        "min_score": 0.3,
        "max_dist_from_anchor_m": 9000,
        "max_travel_minutes": 120,  # for TDTS
    },
    # {
    #     "name": "just coffee_and_food",
    #     "user_text": (
    #         "I want a short-day itinerary with 3–5 enjoyable, wheelchair-accessible coffee or food stops. "
    #         "Please include multiple different locations close to each other."
    #     ),
    #     "constraints": [],
    #     "prefs": ["coffee",],
    #     "categories": ["food"],
    #     "min_score": 0.1,
    #     "max_dist_from_anchor_m": 10000,
    #     "max_travel_minutes": 100,
    # },
    # {
    #     "name": "wheelchair_outdoors_easy_day",
    #     "user_text": (
    #         "I use a wheelchair and prefer open outdoor areas like parks, gardens, and waterfront views. "
    #         "Please avoid steep paths or uneven terrain. I want a peaceful accessible day."
    #     ),
    #     "constraints": [],
    #     "prefs": ["park", "waterfront", "nature"],
    #     "categories": ["park", "attraction", "food"],
    #     "min_score": 0.3,
    #     "max_dist_from_anchor_m": 7000,
    #     "max_travel_minutes": 120,
    # },
    # {
    #     "name": "wheelchair_food_and_sights",
    #     "user_text": (
    #         "I use a wheelchair and want an itinerary focused on accessible restaurants and a few popular sights. "
    #         "Make sure all stops are wheelchair-friendly with smooth access."
    #     ),
    #     "constraints": ["wheelchair"],
    #     "prefs": ["food", "coffee", "easy_sights"],
    #     "categories": ["food", "attraction", "park"],
    #     "min_score": 0.1,
    #     "max_dist_from_anchor_m": 9000,
    #     "max_travel_minutes": 120,
    # },
]

# --------------------------------------------------
# 2) Prompt builder (align with SFT formatting)
# --------------------------------------------------

SYSTEM_PROMPT = (
    "You are an accessibility-aware itinerary planner for San Diego. "
    "You must obey all accessibility constraints, be honest about the data, "
    "and prefer itineraries with short travel distances between stops."
)

def build_eval_user_message(scenario: Dict[str, Any], cand_text: str) -> str:
    constraints = scenario["constraints"]
    if constraints:
        constraints_text = "\n".join(f"- {c}" for c in constraints)
    else:
        constraints_text = "- none"
    return (
        "You are a helpful assistant that plans one-day itineraries in San Diego.\n\n"
        "You MUST:\n"
        "- Use ONLY the places listed below (do NOT invent new locations).\n"
        "- Respect all accessibility needs.\n"
        "- Prefer short travel distances between consecutive stops.\n"
        "- Produce a realistic schedule with non-overlapping times.\n\n"
        "Here is the traveler description:\n"
        f"{scenario['user_text']}\n\n"
        "Accessibility needs:\n"
        f"{constraints_text}\n\n"
        "Below are candidate places you may use in the itinerary:\n"
        f"{cand_text}\n\n"
        "Using only these places, create a coherent one-day itinerary. "
        "Make sure the itinerary is realistic in terms of timing and explicitly respects the accessibility needs.\n\n"
        "Return your answer as a JSON object with a single field \"itinerary\", where the value is a list of stops. "
        "Each stop must have:\n"
        "- \"name\": the place name\n"
        "- \"start_time\": e.g., \"10:00\"\n"
        "- \"end_time\": e.g., \"11:30\"\n"
        "- \"explanation\": a short explanation of why this stop is a good fit, "
        "including accessibility reasoning and, when relevant, short travel distance from the previous stop.\n"
    )

def build_eval_prompt(scenario: Dict[str, Any], cand_text: str) -> str:
    """
    Match the SFT formatting_func style:
    <s>[SYSTEM] ... [/SYSTEM]
    [USER] ... [/USER]
    [ASSISTANT]
    """
    user_msg = build_eval_user_message(scenario, cand_text)
    return (
        f"<s>[SYSTEM]\n{SYSTEM_PROMPT}\n[/SYSTEM]\n"
        f"[USER]\n{user_msg}\n[/USER]\n"
        f"[ASSISTANT]\n"
    )

# --------------------------------------------------
# 3) Helper to run model inference
# --------------------------------------------------

def generate_itinerary_json(prompt: str, max_new_tokens: int = 512) -> Dict[str, Any]:
    model.eval()
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    )
    for k in inputs:
        inputs[k] = inputs[k].to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=False)

    # Extract substring after our prompt
    generated = full_text[len(prompt):]

    # Try to locate JSON block
    try:
        start = generated.index("{")
        end = generated.rindex("}") + 1
        json_str = generated[start:end]
        obj = json.loads(json_str)
    except Exception:
        return {"itinerary": None, "_raw": generated}

    return obj

# --------------------------------------------------
# 4) Evaluation loop
# --------------------------------------------------

results: List[Dict[str, Any]] = []

NUM_SAMPLES_PER_SCENARIO = 3

for scenario in EVAL_SCENARIOS:
    for s_idx in range(NUM_SAMPLES_PER_SCENARIO):
        # 1) RAG: anchor + candidates + candidate text
        try:
            anchor, cands, cand_text = rag_pipeline_for_user(
                user_text=scenario["user_text"],
                anchor_name=None,
                wheelchair_only=False,
                categories=scenario["categories"],
                min_score=scenario["min_score"],
                max_dist_from_anchor_m=scenario["max_dist_from_anchor_m"],
                top_k_semantic=80,
                top_k_final=80,
            )
        except Exception as e:
            print(f"[ERROR] RAG failed for scenario {scenario['name']} sample {s_idx}: {e}")
            results.append({
                "scenario": scenario["name"],
                "sample": s_idx,
                "json_ok": False,
                "num_stops": 0,
                "tdts": None,
                "diversity": None,
                "pref_cov": None,
                "acc": None,
                "comp": None,
                "error": f"RAG error: {e}",
            })
            continue

        # 2) Build prompt + call model
        prompt = build_eval_prompt(scenario, cand_text)
        out_obj = generate_itinerary_json(prompt)
        itinerary = out_obj.get("itinerary")

        json_ok = isinstance(itinerary, list)
        if not json_ok:
            results.append({
                "scenario": scenario["name"],
                "sample": s_idx,
                "json_ok": False,
                "num_stops": 0,
                "tdts": None,
                "diversity": None,
                "pref_cov": None,
                "acc": None,
                "comp": None,
                "error": "Invalid or missing itinerary JSON",
            })
            continue

        # 3) Compute metrics
        max_travel_minutes = scenario.get("max_travel_minutes", 120)
        tdts = compute_tdts(itinerary, max_travel_minutes=max_travel_minutes)
        diversity = compute_diversity_index(itinerary)
        pref_cov = compute_preference_coverage(itinerary, scenario["prefs"])
        acc = compute_access_accuracy(itinerary, scenario["constraints"])
        comp = compute_compliance_rate(itinerary, scenario["constraints"])

        results.append({
            "scenario": scenario["name"],
            "sample": s_idx,
            "json_ok": json_ok,
            "num_stops": len(itinerary),
            "tdts": tdts,
            "diversity": diversity,
            "pref_cov": pref_cov,
            "acc": acc,
            "comp": comp,
            "error": None,
        })

# --------------------------------------------------
# 5) Turn into DataFrame + summary
# --------------------------------------------------

df = pd.DataFrame(results)
print(df)

summary = df.groupby("scenario")[["tdts", "diversity", "pref_cov", "acc", "comp"]].mean()
print("\n=== Scenario-level metric means ===")
print(summary)

[anchor] auto-picked anchor: San Diego Air & Space Museum (category=attraction) from 5 candidates
[rag] retrieved 16 candidates around anchor San Diego Air & Space Museum
[anchor] auto-picked anchor: San Diego Natural History Museum (category=attraction) from 5 candidates
[rag] retrieved 17 candidates around anchor San Diego Natural History Museum
[anchor] auto-picked anchor: San Diego Air & Space Museum (category=attraction) from 5 candidates
[rag] retrieved 16 candidates around anchor San Diego Air & Space Museum
                    scenario  sample  json_ok  num_stops      tdts  diversity  \
0  wheelchair_museums_coffee       0     True          5  0.895920   0.400000   
1  wheelchair_museums_coffee       1     True          6  0.911345   0.333333   
2  wheelchair_museums_coffee       2     True          5  0.895920   0.400000   

   pref_cov  acc  comp error  
0  0.600000  1.0   1.0  None  
1  0.166667  1.0   1.0  None  
2  0.600000  1.0   1.0  None  

=== Scenario-level metric mea

## Model + SFT + RAG + Hybrid-EBR

In [107]:
import json
import random
from typing import List, Dict, Any
import torch
import pandas as pd

# Hyperparameters for EBR
NUM_SAMPLES_PER_SCENARIO = 2       # outer loop (how many prompts per scenario)
NUM_EBR_CANDIDATES = 4             # how many candidate itineraries per prompt

# Energy weights (tune later)
W_DIST  = 1.0   # weight on (1 - TDTS)
W_ACC   = 2.0   # weight on (1 - accessibility)
W_DIV   = 1.0   # weight on (1 - diversity)
W_PREF  = 0.5   # weight on (1 - preference coverage)

# Evaluation scenarios (reuse your previous one)
EVAL_SCENARIOS = [
    {
        "name": "wheelchair_museums_coffee",
        "user_text": (
            "I use a wheelchair and I love art museums and coffee. "
            "I want a relaxed, accessible one-day itinerary in San Diego from 10:00 to 18:00 "
            "with a mix of culture and food."
        ),
        "constraints": [],
        "prefs": ["museum", "art", "coffee"],
        "categories": ["attraction", "food", "park"],
        "min_score": 0.3,
        "max_dist_from_anchor_m": 9000,
        "max_travel_minutes": 120,  # for TDTS
    },
]

# Prompt builder (align with SFT formatting)

SYSTEM_PROMPT = (
    "You are an accessibility-aware itinerary planner for San Diego. "
    "You must obey all accessibility constraints, be honest about the data, "
    "and prefer itineraries with short travel distances between stops."
)

def build_eval_user_message(scenario: Dict[str, Any], cand_text: str) -> str:
    constraints = scenario["constraints"]
    if constraints:
        constraints_text = "\n".join(f"- {c}" for c in constraints)
    else:
        constraints_text = "- none"
    return (
        "You are a helpful assistant that plans one-day itineraries in San Diego.\n\n"
        "You MUST:\n"
        "- Use ONLY the places listed below (do NOT invent new locations).\n"
        "- Respect all accessibility needs.\n"
        "- Prefer short travel distances between consecutive stops.\n"
        "- Produce a realistic schedule with non-overlapping times.\n\n"
        "Here is the traveler description:\n"
        f"{scenario['user_text']}\n\n"
        "Accessibility needs:\n"
        f"{constraints_text}\n\n"
        "Below are candidate places you may use in the itinerary:\n"
        f"{cand_text}\n\n"
        "Using only these places, create a coherent one-day itinerary. "
        "Make sure the itinerary is realistic in terms of timing and explicitly respects the accessibility needs.\n\n"
        "Return your answer as a JSON object with a single field \"itinerary\", where the value is a list of stops. "
        "Each stop must have:\n"
        "- \"name\": the place name\n"
        "- \"start_time\": e.g., \"10:00\"\n"
        "- \"end_time\": e.g., \"11:30\"\n"
        "- \"explanation\": a short explanation of why this stop is a good fit, "
        "including accessibility reasoning and, when relevant, short travel distance from the previous stop.\n"
    )

def build_eval_prompt(scenario: Dict[str, Any], cand_text: str) -> str:
    """
    Match the SFT formatting_func style:
    <s>[SYSTEM] ... [/SYSTEM]
    [USER] ... [/USER]
    [ASSISTANT]
    """
    user_msg = build_eval_user_message(scenario, cand_text)
    return (
        f"<s>[SYSTEM]\n{SYSTEM_PROMPT}\n[/SYSTEM]\n"
        f"[USER]\n{user_msg}\n[/USER]\n"
        f"[ASSISTANT]\n"
    )

# Single generation helper (with sampling) 

def generate_itinerary_json(prompt: str,
                            max_new_tokens: int = 512,
                            do_sample: bool = True,
                            temperature: float = 0.7,
                            top_p: float = 0.9) -> Dict[str, Any]:
    
    model.eval()
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048,
    )
    for k in inputs:
        inputs[k] = inputs[k].to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_p=top_p,
            pad_token_id=tokenizer.eos_token_id,
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
    generated = full_text[len(prompt):]

    try:
        start = generated.index("{")
        end = generated.rindex("}") + 1
        json_str = generated[start:end]
        obj = json.loads(json_str)
    except Exception:
        return {"itinerary": None, "_raw": generated}

    return obj


def compute_energy_from_metrics(itinerary: List[Dict[str, Any]], scenario: Dict[str, Any]):
    """
    E = W_DIST * (1 - TDTS)
      + W_ACC  * (1 - accessibility)
      + W_DIV  * (1 - diversity)
      + W_PREF * (1 - preference coverage)

    Where accessibility = average of accuracy + compliance.
    """
    max_travel_minutes = scenario.get("max_travel_minutes", 120)
    prefs       = scenario.get("prefs", [])
    constraints = scenario.get("constraints", [])

    # metrics
    tdts     = compute_tdts(itinerary, max_travel_minutes=max_travel_minutes)
    div      = compute_diversity_index(itinerary)
    pref_cov = compute_preference_coverage(itinerary, prefs)
    acc      = compute_access_accuracy(itinerary, constraints)
    comp     = compute_compliance_rate(itinerary, constraints)

    # safe defaults when metric is None
    tdts_val = tdts if tdts is not None else 1.0
    div_val  = div if div is not None else 1.0
    pref_val = pref_cov if pref_cov is not None else 1.0
    acc_val  = acc if acc is not None else 1.0
    comp_val = comp if comp is not None else 1.0

    acc_combined = 0.5 * acc_val + 0.5 * comp_val

    dist_penalty  = 1.0 - tdts_val
    acc_penalty   = 1.0 - acc_combined
    div_penalty   = 1.0 - div_val
    prefs_penalty = 1.0 - pref_val

    energy = (
        W_DIST * dist_penalty
        + W_ACC * acc_penalty
        + W_DIV * div_penalty
        + W_PREF * prefs_penalty
    )

    metrics = {
        "tdts": tdts,
        "diversity": div,
        "pref_cov": pref_cov,
        "acc": acc,
        "comp": comp,
    }
    return energy, metrics

# Evaluation loop with EBR

results: List[Dict[str, Any]] = []

for scenario in EVAL_SCENARIOS:
    for s_idx in range(NUM_SAMPLES_PER_SCENARIO):
        # RAG: anchor + candidates + candidate text
        try:
            anchor, cands, cand_text = rag_pipeline_for_user(
                user_text=scenario["user_text"],
                anchor_name=None,
                wheelchair_only=("wheelchair" in scenario["constraints"]),
                categories=scenario["categories"],
                min_score=scenario["min_score"],
                max_dist_from_anchor_m=scenario["max_dist_from_anchor_m"],
                top_k_semantic=80,
                top_k_final=80,
            )
            print(
                f"[anchor] {scenario['name']} sample {s_idx}: "
                f"{anchor['_display_name']} (category={anchor['_category']})"
            )
        except Exception as e:
            print(f"[ERROR] RAG failed for scenario {scenario['name']} sample {s_idx}: {e}")
            results.append({
                "scenario": scenario["name"],
                "sample": s_idx,
                "json_ok": False,
                "num_stops": 0,
                "tdts": None,
                "diversity": None,
                "pref_cov": None,
                "acc": None,
                "comp": None,
                "energy": None,
                "error": f"RAG error: {e}",
            })
            continue

        # Build prompt
        prompt = build_eval_prompt(scenario, cand_text)

        # Sample multiple candidate itineraries + compute energy
        candidate_records = []
        for k in range(NUM_EBR_CANDIDATES):
            out_obj = generate_itinerary_json(
                prompt,
                max_new_tokens=512,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
            )
            itinerary = out_obj.get("itinerary")

            if not isinstance(itinerary, list) or len(itinerary) == 0:
                print(f"[EBR] scenario={scenario['name']} sample={s_idx} cand={k}: INVALID itinerary")
                continue

            energy, metrics = compute_energy_from_metrics(itinerary, scenario)

            # Print per-candidate info
            print(
                f"[EBR] scenario={scenario['name']} sample={s_idx} cand={k}: "
                f"energy={energy:.3f}, tdts={metrics['tdts']}, "
                f"div={metrics['diversity']}, pref_cov={metrics['pref_cov']}, "
                f"acc={metrics['acc']}, comp={metrics['comp']}"
            )

            candidate_records.append({
                "k": k,
                "itinerary": itinerary,
                "energy": energy,
                **metrics,
            })

        if not candidate_records:
            # no valid candidates from the model
            results.append({
                "scenario": scenario["name"],
                "sample": s_idx,
                "json_ok": False,
                "num_stops": 0,
                "tdts": None,
                "diversity": None,
                "pref_cov": None,
                "acc": None,
                "comp": None,
                "energy": None,
                "error": "no_valid_candidate",
            })
            continue

        # EBR: pick lowest-energy itinerary
        best = min(candidate_records, key=lambda x: x["energy"])
        best_it = best["itinerary"]

        print(
            f"[EBR] scenario={scenario['name']} sample={s_idx}: "
            f"chosen cand={best['k']} with energy={best['energy']:.3f}"
        )

        results.append({
            "scenario": scenario["name"],
            "sample": s_idx,
            "json_ok": True,
            "num_stops": len(best_it),
            "tdts": best["tdts"],
            "diversity": best["diversity"],
            "pref_cov": best["pref_cov"],
            "acc": best["acc"],
            "comp": best["comp"],
            "energy": best["energy"],
            "error": None,
        })

# DataFrame + summary

df_ebr = pd.DataFrame(results)
print(df_ebr)

summary_ebr = df_ebr.groupby("scenario")[["tdts", "diversity", "pref_cov", "acc", "comp", "energy"]].mean()
print("\n=== Scenario-level metric means (Hybrid EBR) ===")
print(summary_ebr)

[anchor] auto-picked anchor: San Diego Museum of Art (category=attraction) from 5 candidates
[rag] retrieved 17 candidates around anchor San Diego Museum of Art
[anchor] wheelchair_museums_coffee sample 0: San Diego Museum of Art (category=attraction)
[EBR] scenario=wheelchair_museums_coffee sample=0 cand=0: energy=1.030, tdts=0.8869104665579375, div=0.3333333333333333, pref_cov=0.5, acc=1.0, comp=1.0
[EBR] scenario=wheelchair_museums_coffee sample=0 cand=1: energy=1.007, tdts=0.8934327314387197, div=0.4, pref_cov=0.4, acc=1.0, comp=1.0
[EBR] scenario=wheelchair_museums_coffee sample=0 cand=2: energy=0.858, tdts=0.9756898920138074, div=0.16666666666666666, pref_cov=1.0, acc=1.0, comp=1.0
[EBR] scenario=wheelchair_museums_coffee sample=0 cand=3: energy=0.993, tdts=0.9073791870718576, div=0.4, pref_cov=0.4, acc=1.0, comp=1.0
[EBR] scenario=wheelchair_museums_coffee sample=0: chosen cand=2 with energy=0.858
[anchor] auto-picked anchor: Museum of Contemporary Art San Diego (category=attrac